# EfficientRAG Training on V100

Запускай ячейки сверху вниз. Время на V100: ~15-20 минут целиком.

## 1. Проверка GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else '')

## 2. Установка зависимостей

In [ ]:
# Не трогаем уже установленный torch (в Colab/Kaggle/cloud V100 он с CUDA).
# Если torch отсутствует — ставим cu118 wheel под V100 (sm_70).
import importlib.util, subprocess, sys

if importlib.util.find_spec('torch') is None:
    print('torch не найден, ставлю CUDA 11.8 wheel под V100...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
        'torch==2.2.2'])
else:
    import torch
    print(f'torch {torch.__version__} уже установлен (CUDA={torch.cuda.is_available()}) — не трогаю')

# Все остальные зависимости — НЕ переустанавливаем torch.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.46,<4.50',
    'datasets>=2.18',
    'numpy<2.0',
    'scikit-learn',
    'huggingface_hub>=0.24',
    'sentencepiece>=0.2.0',
    'protobuf>=3.20,<5',
    'accelerate>=0.30',
    'safetensors>=0.4',
])
print('OK: deps installed')

## 3. HuggingFace токен

**Не вписывай токен прямо в ячейку!** Используй один из способов:

1. Переменную окружения: запусти в терминале `export HF_TOKEN=hf_xxx` ДО старта Jupyter
2. Файл `.env` в корне проекта (он в `.gitignore`)
3. `huggingface-cli login` в терминале (сохранит в `~/.cache/huggingface/token`)

In [ ]:
import os
from huggingface_hub import login

# Если в окружении нет HF_TOKEN, попробуем .env
if not os.environ.get('HF_TOKEN'):
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

token = os.environ.get('HF_TOKEN')
if not token:
    raise RuntimeError('HF_TOKEN не найден. Установи через export HF_TOKEN=... или .env')

login(token=token)
print('Logged in')

## 4. Скачать тренировочные данные

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs('data/efficient_rag/labeler', exist_ok=True)
os.makedirs('data/efficient_rag/filter', exist_ok=True)

p = hf_hub_download(repo_id='Necent/efficientrag-labeler-training-data',
                    filename='train.jsonl', repo_type='dataset')
shutil.copy(p, 'data/efficient_rag/labeler/train.jsonl')

p = hf_hub_download(repo_id='Necent/efficientrag-filter-training-data',
                    filename='train.jsonl', repo_type='dataset')
shutil.copy(p, 'data/efficient_rag/filter/train.jsonl')

!wc -l data/efficient_rag/labeler/train.jsonl data/efficient_rag/filter/train.jsonl

## 5. Обучение Labeler (~10-15 мин на V100)

In [ ]:
!python -m EfficientRAG.training.train_labeler \
    --train_data data/efficient_rag/labeler/train.jsonl \
    --output_dir checkpoints/efficientrag_labeler \
    --model_name microsoft/mdeberta-v3-base \
    --num_labels 2 \
    --max_length 384 \
    --epochs 2 \
    --batch_size 32 \
    --lr 5e-6 \
    --warmup_steps 200

## 6. Обучение Filter (~2-3 мин на V100)

In [ ]:
!python -m EfficientRAG.training.train_filter \
    --train_data data/efficient_rag/filter/train.jsonl \
    --output_dir checkpoints/efficientrag_filter \
    --model_name microsoft/mdeberta-v3-base \
    --max_length 128 \
    --epochs 2 \
    --batch_size 32 \
    --lr 1e-5 \
    --warmup_steps 200

## 7. Загрузить модели на HuggingFace

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_folder(
    folder_path='checkpoints/efficientrag_labeler',
    repo_id='Necent/efficientrag-labeler-mdeberta-v3-base',
    repo_type='model',
)
print('Labeler uploaded!')

api.upload_folder(
    folder_path='checkpoints/efficientrag_filter',
    repo_id='Necent/efficientrag-filter-mdeberta-v3-base',
    repo_type='model',
)
print('Filter uploaded!')

print('\nDone! Models:')
print('  https://huggingface.co/Necent/efficientrag-labeler-mdeberta-v3-base')
print('  https://huggingface.co/Necent/efficientrag-filter-mdeberta-v3-base')